In [15]:
from beir.datasets.data_loader import GenericDataLoader
from rank_bm25 import BM25Okapi

# Load data (same as Phase 1)
data_path = "data/raw/msmarco"
corpus, queries, qrels = GenericDataLoader(data_path).load(split="dev")

# Build filtered corpus (same logic as before)
relevant_doc_ids = set()
for qid, docs in qrels.items():
    for d in docs:
        relevant_doc_ids.add(d)

filtered_corpus = {d: corpus[d] for d in relevant_doc_ids}

# IMPORTANT: doc_ids mapping
doc_ids = list(filtered_corpus.keys())

# BM25 index
docs_text = [filtered_corpus[d]["text"] for d in doc_ids]
tokenized_docs = [text.lower().split() for text in docs_text]

bm25 = BM25Okapi(tokenized_docs)


100%|█████████████████████████████| 8841823/8841823 [00:16<00:00, 542402.63it/s]


In [16]:
ranked_results = pd.read_csv("data/processed/ltr_ranked_results.csv")
ranked_results["query_id"] = ranked_results["query_id"].astype(str)
ranked_results["doc_id"] = ranked_results["doc_id"].astype(str)


In [21]:
import numpy as np
import pandas as pd

BM25_THRESHOLD = 8.0   # tune later

hybrid_rows = []

for qid, group in ranked_results.groupby("query_id"):
    qid = str(qid)
    query_text = queries[qid]

    # BM25 scores for candidate docs
    bm25_scores = []
    for doc_id in group["doc_id"]:
        idx = doc_ids.index(str(doc_id))
        bm25_scores.append(
            bm25.get_scores(query_text.lower().split())[idx]
        )

    bm25_scores = np.array(bm25_scores)
    max_bm25 = bm25_scores.max()

    temp = group.copy()

    if max_bm25 >= BM25_THRESHOLD:
        # TRUST BM25 (exact-match query)
        temp["hybrid_score"] = bm25_scores
    else:
        # TRUST LTR (semantic / vague query)
        temp["hybrid_score"] = temp["score"].values

    hybrid_rows.append(temp)

hybrid_df = pd.concat(hybrid_rows)

hybrid_df = hybrid_df.sort_values(
    ["query_id", "hybrid_score"],
    ascending=[True, False]
)

hybrid_df.head()


,query_id,doc_id,tfidf,bm25,common_terms,overlap_ratio,label,score,hybrid_score
315,1020724,7232727,0.191040,14.672613,2,0.4,1,0.260688,14.672613
317,1020724,1152650,0.081147,12.062003,3,0.6,0,0.062296,12.062003
320,1020724,7679571,0.038923,9.769264,3,0.6,0,0.035551,9.769264
321,1020724,7322168,0.040431,9.640972,3,0.6,0,0.035379,9.640972
316,1020724,7937668,0.059996,9.048922,2,0.4,0,0.063827,9.048922


In [22]:
from ranx import Qrels, Run, evaluate

qrels_hybrid = {}
run_hybrid = {}

for qid, group in hybrid_df.groupby("query_id"):
    qid = str(qid)
    qrels_hybrid[qid] = {}
    run_hybrid[qid] = {}

    for _, row in group.iterrows():
        qrels_hybrid[qid][str(row["doc_id"])] = int(row["label"])
        run_hybrid[qid][str(row["doc_id"])] = float(row["hybrid_score"])

metrics_hybrid = evaluate(
    Qrels(qrels_hybrid),
    Run(run_hybrid),
    ["ndcg@10", "map"]
)

metrics_hybrid


{'ndcg@10': np.float64(0.7377655384806957),
 'map': np.float64(0.6617989417989418)}

I experimented with both weighted and gated hybrid ranking strategies.
However, since my LTR model already incorporated BM25-based features, hybrid fusion did not add complementary signal and degraded performance.
This showed that hybrid methods are not universally beneficial and must be justified by orthogonal signals.